# 🛡️ Real-Time Threat Detection — Analytical Query Visualizations
**Stack:** VoltDB → Pub/Sub → BigQuery + Dataform (MaxMind geo enrichment)

This notebook connects to BigQuery and renders interactive visualizations for each of the six analytical queries described in the threat detection demo article.

---

## 0 · Setup & BigQuery Connection

In [2]:
# Install required packages (run once)
# !pip install google-cloud-bigquery db-dtypes plotly pandas kaleido

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import bigquery

# ── Configuration ────────────────────────────────────────────────────────────
PROJECT_ID = "<your-gcp-project>"   # ← replace with your GCP project
DATASET    = "<your-dataset>"       # ← replace with your BigQuery dataset

TABLE_TXN  = f"`{PROJECT_ID}.{DATASET}.threat_transactions`"
TABLE_GEO  = f"`{PROJECT_ID}.{DATASET}.threat_transactions_geo`"

# ── Colour palette (matches article brand) ───────────────────────────────────
RULE_COLORS = {
    "SUBNET_RATE":    "#c62828",   # threat red
    "VELOCITY_BURST": "#e65100",   # orange
    "HIGH_SPEND":     "#f9a825",   # amber
    "VALIDATION":     "#5f6368",   # grey
}
DEFAULT_COLOR = "#1a73e8"           # volt blue

# ── BigQuery client ──────────────────────────────────────────────────────────
# When running in Google Colab / Vertex AI Workbench, ADC (Application Default
# Credentials) is used automatically.  Outside GCP, set GOOGLE_APPLICATION_CREDENTIALS
# or run `gcloud auth application-default login` first.
client = bigquery.Client(project=PROJECT_ID)
print(f"Connected to project: {PROJECT_ID}")

In [4]:
def run_query(sql: str) -> pd.DataFrame:
    """Execute a BigQuery SQL string and return a pandas DataFrame."""
    return client.query(sql).to_dataframe()

def rule_color_list(series: pd.Series) -> list:
    """Map a Series of rule names to hex colours."""
    return [RULE_COLORS.get(r, DEFAULT_COLOR) for r in series]

---
## Query 1 · Threat Summary — All Blocked Transactions
> Returns every blocked transaction with full context — no JOINs needed because the data was denormalised at publish time.

In [5]:
sql_q1 = f"""
SELECT
  txn_time, source_ip, rule_name, reason,
  amount, account_name, merchant_name, merchant_category
FROM {TABLE_TXN}
WHERE accepted = 0
ORDER BY txn_time DESC
"""
df_q1 = run_query(sql_q1)
print(f"Blocked transactions: {len(df_q1):,}")
df_q1.head()

Blocked transactions: 803


,txn_time,source_ip,rule_name,reason,amount,account_name,merchant_name,merchant_category
0,2026-03-19 16:38:05.832000+00:00,214.0.1.5,VALIDATION,Account Disabled,86.10,Diana Prince,GroceryKing,Food
1,2026-03-19 16:38:04.832000+00:00,89.160.20.140,VALIDATION,Account Disabled,61.95,Diana Prince,FashionHub,Retail
2,2026-03-19 16:38:03.832000+00:00,89.160.20.129,VALIDATION,Invalid Merchant,139.32,Alice Johnson,None,None
3,2026-03-19 16:38:03.832000+00:00,2.125.160.219,VALIDATION,Account Disabled,89.38,Diana Prince,QuickBite Deli,Food
4,2026-03-19 16:38:02.832000+00:00,2.125.160.218,VALIDATION,Account Disabled,47.44,Diana Prince,ElectroMart,Retail


In [6]:
# ── Timeline scatter: every blocked txn as a dot ──────────────────────────────
fig_q1 = px.scatter(
    df_q1,
    x="txn_time",
    y="rule_name",
    color="rule_name",
    color_discrete_map=RULE_COLORS,
    size="amount",
    size_max=22,
    hover_data={
        "txn_time": True,
        "source_ip": True,
        "amount": True,
        "account_name": True,
        "merchant_name": True,
        "reason": True,
        "rule_name": False,
    },
    title="Blocked Transactions Timeline  (dot size = transaction amount)",
    labels={"txn_time": "Transaction Time", "rule_name": "Threat Rule"},
)
fig_q1.update_layout(
    height=420,
    plot_bgcolor="#f8f9fa",
    paper_bgcolor="white",
    legend_title_text="Rule",
    font=dict(family="Segoe UI, sans-serif", size=13),
    title_font_size=16,
)
fig_q1.update_xaxes(showgrid=True, gridcolor="#dadce0")
fig_q1.update_yaxes(showgrid=False)
fig_q1.show()

---
## Query 2 · Threats by Rule — Which Rules Fire Most?
> Groups by rule and date, surfacing threat counts, affected accounts, unique IPs, and total blocked amount.

In [7]:
sql_q2 = f"""
SELECT
  rule_name,
  DATE(txn_time) AS txn_date,
  COUNT(*)                      AS threat_count,
  COUNT(DISTINCT account_id)    AS affected_accounts,
  COUNT(DISTINCT source_ip)     AS unique_ips,
  SUM(amount)                   AS total_amount
FROM {TABLE_TXN}
WHERE accepted = 0 AND rule_name != 'VALIDATION'
GROUP BY rule_name, DATE(txn_time)
ORDER BY txn_date DESC, threat_count DESC
"""
df_q2 = run_query(sql_q2)
df_q2["txn_date"] = pd.to_datetime(df_q2["txn_date"])
df_q2.head(12)

,rule_name,txn_date,threat_count,affected_accounts,unique_ips,total_amount
0,TXN_SUMMARY_30SEC,2026-03-19,630,4,197,19709.42
1,TXN_SUMMARY_1MIN,2026-03-19,32,4,4,36800.00
2,TXN_SUMMARY_30SEC,2026-03-18,88,4,67,2793.00
3,SUBNET_RATE,2026-03-18,15,4,10,474.00
4,TXN_SUMMARY_1MIN,2026-03-18,2,1,1,2700.00


In [8]:
# ── Four-panel dashboard: count, amount, accounts, IPs ───────────────────────
fig_q2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Threat Count by Rule & Date",
        "Total Blocked Amount ($) by Rule & Date",
        "Affected Accounts by Rule & Date",
        "Unique Source IPs by Rule & Date",
    ),
    shared_xaxes=True,
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

metrics = [
    ("threat_count",      1, 1),
    ("total_amount",      1, 2),
    ("affected_accounts", 2, 1),
    ("unique_ips",        2, 2),
]

seen_rules = set()
for metric, row, col in metrics:
    for rule, grp in df_q2.groupby("rule_name"):
        show_legend = (metric == "threat_count") and (rule not in seen_rules)
        seen_rules.add(rule)
        fig_q2.add_trace(
            go.Bar(
                x=grp["txn_date"],
                y=grp[metric],
                name=rule,
                marker_color=RULE_COLORS.get(rule, DEFAULT_COLOR),
                legendgroup=rule,
                showlegend=show_legend,
                hovertemplate=f"<b>{rule}</b><br>Date: %{{x|%Y-%m-%d}}<br>{metric}: %{{y:,.0f}}<extra></extra>",
            ),
            row=row, col=col,
        )

fig_q2.update_layout(
    height=560,
    barmode="group",
    title_text="Threats by Rule — Daily Breakdown",
    title_font_size=16,
    plot_bgcolor="#f8f9fa",
    paper_bgcolor="white",
    legend_title_text="Rule",
    font=dict(family="Segoe UI, sans-serif", size=12),
)
fig_q2.update_xaxes(showgrid=False)
fig_q2.update_yaxes(showgrid=True, gridcolor="#dadce0")
fig_q2.show()

---
## Query 5 · Threat Heatmap — Geographic Bubble Map
> Lat/lon coordinates from MaxMind, bubble sized by threat count and coloured by rule type. Each point is an attack origin.

In [14]:
sql_q5 = f"""
SELECT
  latitude, longitude, country_name, city_name, rule_name,
  COUNT(*) AS threat_count
FROM {TABLE_GEO}
WHERE accepted = 0
  AND latitude  IS NOT NULL
  AND rule_name != 'VALIDATION'
GROUP BY latitude, longitude, country_name, city_name, rule_name
ORDER BY threat_count DESC
"""
df_q5 = run_query(sql_q5)
df_q5.head()

,latitude,longitude,country_name,city_name,rule_name,threat_count
0,51.5142,-0.0931,United Kingdom,London,TXN_SUMMARY_30SEC,255
1,32.7405,-117.0935,United States,San Diego,TXN_SUMMARY_30SEC,220
2,43.8800,125.3228,China,Changchun,TXN_SUMMARY_30SEC,220
3,47.2513,-122.3149,United States,Milton,TXN_SUMMARY_30SEC,13
4,58.4167,15.6167,Sweden,Linköping,TXN_SUMMARY_30SEC,10


In [15]:
# ── Bubble map — scatter_geo ───────────────────────────────────────────────────
fig_q5 = px.scatter_geo(
    df_q5,
    lat="latitude",
    lon="longitude",
    size="threat_count",
    color="rule_name",
    color_discrete_map=RULE_COLORS,
    hover_name="city_name",
    hover_data={
        "country_name": True,
        "rule_name": True,
        "threat_count": True,
        "latitude": False,
        "longitude": False,
    },
    size_max=50,
    projection="natural earth",
    title="Threat Heatmap — Geographic Origin  (bubble size = threat count)",
    labels={"rule_name": "Rule", "threat_count": "Threat Count"},
)
fig_q5.update_layout(
    height=500,
    paper_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=12),
    legend_title_text="Threat Rule",
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor="#dadce0",
        bgcolor="#f8f9fa",
        landcolor="#e8f0fe",
        showland=True,
        showocean=True,
        oceancolor="#e3f2fd",
        showcountries=True,
        countrycolor="#c5cae9",
    ),
)
fig_q5.show()

---
## Query 6 · Account Risk Profile
> For every account: total threats triggered, how many distinct rules fired, how many different source IPs attacked it, and its overall acceptance rate. High risk = high threats + low accept_pct + multiple rules.

In [11]:
sql_q6 = f"""
SELECT
  account_id, account_name,
  COUNT(*)                                            AS total_threats,
  COUNT(DISTINCT rule_name)                           AS distinct_rules,
  COUNT(DISTINCT source_ip)                           AS distinct_ips,
  ROUND(
    100.0 * SUM(CASE WHEN accepted = 1 THEN 1 ELSE 0 END) / COUNT(*),
  1)                                                  AS accept_pct
FROM {TABLE_TXN}
GROUP BY account_id, account_name
ORDER BY total_threats DESC
"""
df_q6 = run_query(sql_q6)
df_q6.head(10)

,account_id,account_name,total_threats,distinct_rules,distinct_ips,accept_pct
0,1,Alice Johnson,557,5,157,61.4
1,2,Bob Smith,516,4,146,63.2
2,3,Charlie Brown,496,4,151,62.1
3,5,Eve Wilson,496,4,149,62.1
4,4,Diana Prince,22,1,5,0.0


In [12]:
# ── Derive a composite risk score for colour coding ───────────────────────────
# risk_score = total_threats × distinct_rules × (1 - accept_pct/100)
df_q6["risk_score"] = (
    df_q6["total_threats"] *
    df_q6["distinct_rules"] *
    (1 - df_q6["accept_pct"] / 100)
).round(2)

df_q6["risk_label"] = pd.cut(
    df_q6["risk_score"],
    bins=[-1, 0, 5, 20, float("inf")],
    labels=["Clean", "Low", "Medium", "High"],
)
risk_palette = {"Clean": "#34a853", "Low": "#fbbc04", "Medium": "#e65100", "High": "#c62828"}

In [13]:
# ── Bubble scatter: total_threats vs accept_pct, size = distinct_ips ──────────
fig_q6_bubble = px.scatter(
    df_q6,
    x="accept_pct",
    y="total_threats",
    size="distinct_ips",
    color="risk_label",
    color_discrete_map=risk_palette,
    hover_name="account_name",
    hover_data={
        "account_id": True,
        "total_threats": True,
        "distinct_rules": True,
        "distinct_ips": True,
        "accept_pct": True,
        "risk_score": True,
        "risk_label": False,
    },
    size_max=40,
    title="Account Risk Profile  (bubble size = distinct attacking IPs)",
    labels={
        "accept_pct": "Acceptance Rate (%)",
        "total_threats": "Total Threats",
        "risk_label": "Risk Level",
    },
    category_orders={"risk_label": ["High", "Medium", "Low", "Clean"]},
)

# Add quadrant reference lines
fig_q6_bubble.add_vline(
    x=80, line_dash="dot", line_color="#9e9e9e",
    annotation_text="80% accept rate", annotation_position="top right",
)
fig_q6_bubble.add_hline(
    y=df_q6["total_threats"].median(), line_dash="dot", line_color="#9e9e9e",
    annotation_text="median threats", annotation_position="right",
)

fig_q6_bubble.update_layout(
    height=500,
    plot_bgcolor="#f8f9fa",
    paper_bgcolor="white",
    font=dict(family="Segoe UI, sans-serif", size=12),
    legend_title_text="Risk Level",
)
fig_q6_bubble.update_xaxes(range=[-2, 102], showgrid=True, gridcolor="#dadce0")
fig_q6_bubble.update_yaxes(showgrid=True, gridcolor="#dadce0")
fig_q6_bubble.show()